In [1]:
import requests
import pandas as pd

In [ ]:
url = "https://places.googleapis.com/v1/places:searchText"

API KEY = ''

In [9]:
def search_all_branches(query, api_key):
    url = "https://places.googleapis.com/v1/places:searchText"
    
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,nextPageToken"
    }
    
    all_branches = []
    next_page_token = None
    
    while True:
        body = {
            "textQuery": query,
            "pageSize": 20,
            # restrict the map to metro manila (the capital)
            "locationRestriction": {
                "rectangle": {
                    "low": {"latitude": 14.3, "longitude": 120.9},
                    # low = Metro Manila's southwest corner
                    "high": {"latitude": 14.8, "longitude": 121.2}
                    # high = Metro Manila's northeast corner
                }
            }
        }
        
        if next_page_token:
            body["pageToken"] = next_page_token
        
        response = requests.post(url, headers=headers, json=body)
        data = response.json()
        
        branches = data.get("places", [])
        all_branches.extend(branches)
        print(f"Fetched {len(branches)} branches, total so far: {len(all_branches)}")
        
        next_page_token = data.get("nextPageToken")
        if not next_page_token:
            break
            
        # pause between requests to avoid rate limits
        import time
        time.sleep(2)
    
    return all_branches

In [10]:
# Loop all cities in Metro Manila
ncr_cities = [
    "Manila", "Quezon City", "Caloocan", "Makati", "Pasig",
    "Taguig", "Parañaque", "Las Piñas", "Muntinlupa", "Marikina",
    "Mandaluyong", "Valenzuela", "Navotas", "Malabon", "Pasay",
    "San Juan", "Pateros"
]

def search_by_city(brand, cities, api_key):
    all_branches = []
    
    for city in cities:
        query = f"{brand} {city} Metro Manila Philippines"
        branches = search_all_branches(query, api_key)
        all_branches.extend(branches)
        time.sleep(2)
    
    return all_branches

jollibee_branches = search_by_city("Jollibee", ncr_cities, API_KEY)
mcdo_branches = search_by_city("McDonald's", ncr_cities, API_KEY)

print(f"\nTotal Jollibee branches: {len(jollibee_branches)}")
print(f"Total McDonald's branches: {len(mcdo_branches)}")

Fetched 20 branches, total so far: 20
Fetched 20 branches, total so far: 40
Fetched 20 branches, total so far: 60
Fetched 20 branches, total so far: 20
Fetched 20 branches, total so far: 40
Fetched 20 branches, total so far: 60
Fetched 20 branches, total so far: 20
Fetched 20 branches, total so far: 40
Fetched 16 branches, total so far: 56
Fetched 20 branches, total so far: 20
Fetched 12 branches, total so far: 32
Fetched 20 branches, total so far: 20
Fetched 20 branches, total so far: 40
Fetched 20 branches, total so far: 60
Fetched 20 branches, total so far: 20
Fetched 13 branches, total so far: 33
Fetched 20 branches, total so far: 20
Fetched 5 branches, total so far: 25
Fetched 19 branches, total so far: 19
Fetched 20 branches, total so far: 20
Fetched 1 branches, total so far: 21
Fetched 20 branches, total so far: 20
Fetched 14 branches, total so far: 34
Fetched 20 branches, total so far: 20
Fetched 1 branches, total so far: 21
Fetched 20 branches, total so far: 20
Fetched 11 bran

In [11]:
#Remove duplicates

def deduplicate(branches):
    seen = set()
    unique = []

    for branch in branches:
        #Round to four decimal places
        lat = round(branch['location']['latitude'], 4)
        lng = round(branch['location']['longitude'], 4)
        key = (lat, lng)
        
        if key not in seen:
            seen.add(key)
            unique.append(branch)
    
    return unique

jollibee_unique = deduplicate(jollibee_branches)
mcdo_unique = deduplicate(mcdo_branches)

print(f"Jollibee (deduplicated): {len(jollibee_unique)}")
print(f"McDonald's (deduplicated): {len(mcdo_unique)}")

Jollibee (deduplicated): 332
McDonald's (deduplicated): 247


In [12]:
#Format the dataframe
def branches_to_df(branches, brand):
    rows = []
    for branch in branches:
        rows.append({
            "brand": brand,
            "name": branch["displayName"]["text"],
            "address": branch["formattedAddress"],
            "latitude": branch["location"]["latitude"],
            "longitude": branch["location"]["longitude"]
        })
    return pd.DataFrame(rows)

jollibee_df = branches_to_df(jollibee_unique, "Jollibee")
mcdo_df = branches_to_df(mcdo_unique, "McDonald's")

df = pd.concat([jollibee_df, mcdo_df], ignore_index=True)

print(df.shape)
print(df.head())

(579, 5)
      brand                       name  \
0  Jollibee                   Jollibee   
1  Jollibee  Jollibee Abad Santos Ave.   
2  Jollibee                   Jollibee   
3  Jollibee                   Jollibee   
4  Jollibee            Jollibee Banawe   

                                             address   latitude   longitude  
0  1038 Edsa E. Congressional, Bago Bantay, Quezo...  14.657793  121.021055  
1  3039 Jose Abad Santos, Tondo, Manila, Metro Ma...  14.632186  120.980317  
2  Bukidnon Street, corner Antique, Bago Bantay, ...  14.659821  121.025850  
3  858 Quezon Ave, Diliman, Quezon City, Metro Ma...  14.631292  121.017197  
4  424 Banawe St, Bgy, cor Ma Clara, Santa Mesa H...  14.627703  121.004315  


In [13]:
ncr_cities = [
    "Manila", "Quezon City", "Caloocan", "Makati", "Pasig",
    "Taguig", "Parañaque", "Las Piñas", "Muntinlupa", "Marikina",
    "Mandaluyong", "Valenzuela", "Navotas", "Malabon", "Pasay",
    "San Juan", "Pateros"
]

def extract_city(address):
    # Remove "Metro Manila" to avoid matching with "Manila"
    cleaned = address.replace("Metro Manila", "")
    
    for city in ncr_cities:
        if city in cleaned:
            return city
    return "Unknown"

df["city"] = df["address"].apply(extract_city)

print(df["city"].value_counts())
print(f"\nUnknown: {(df['city'] == 'Unknown').sum()}")

city
Quezon City    124
Manila          62
Pasig           48
Caloocan        42
Makati          40
Taguig          38
Parañaque       38
Pasay           36
Valenzuela      22
Las Piñas       22
Marikina        22
Mandaluyong     21
Muntinlupa      19
Unknown         17
San Juan        11
Malabon         11
Navotas          4
Pateros          2
Name: count, dtype: int64

Unknown: 17


In [15]:
#Check the unknown locations

print(df[df["city"] == "Unknown"]["address"].tolist())

['UPPER GROUND SM CITY SAN LAZARO F. HUERTAS ROAD, EXTENSION, CORNER Lacson Ave, Barangay 350, Santa Cruz, City, 1008 Metro Manila, Philippines', 'Quirino Hwy, NOVALICHES, GULOD, Metro Manila, Philippines', 'ORTIGAS AVENUE, EXTENSION, CORNER C. Raymundo Ave, ROSARIO, 1609 Metro Manila, Philippines', 'Ortigas Ave. ext, cor Valley Golf Road, Cainta, 1900 Rizal, Philippines', 'RD 3 Commercial Complex, A.Bonifacio Ave, Cainta, 1900 Rizal, Philippines', 'H44H+296, Hwy 2000, Taytay, 1920 Rizal, Philippines', "Level 1 Robinson's Place, Ortigas Ave Ext, Cainta, 1900 Rizal, Philippines", 'Petron Slex Southbound, San Antonio, San Pedro, Laguna, Philippines', 'Pacita Complex, National Highway, Brgy, Nueva, San Pedro, Laguna, Philippines', 'Town & Country Executive Village, 2B Molave, Antipolo, 1800 Rizal, Philippines', '9 A.Bonifacio Ave, Cainta, 1900 Rizal, Philippines', '6037 A.Bonifacio Ave, Cainta, 1900 Rizal, Philippines', 'Ortigas Ave Ext, Cainta, 1900 Rizal, Philippines', 'Brookside Dr, Ca

In [17]:
#Remove locations outside Metro Manila, and fix those inside.

def fix_unknown_city(address):
    if "Santa Cruz" in address and "Metro Manila" in address:
        return "Manila"
    if "NOVALICHES" in address or "GULOD" in address:
        return "Quezon City"
    if "ROSARIO" in address and "Metro Manila" in address:
        return "Pasig"
    return "Unknown"

df.loc[df["city"] == "Unknown", "city"] = df.loc[df["city"] == "Unknown", "address"].apply(fix_unknown_city)

# Drop remaining unknowns
df = df[df["city"] != "Unknown"].reset_index(drop=True)

print(f"Total branches: {len(df)}")
print(df["city"].value_counts())

Total branches: 565
city
Quezon City    125
Manila          63
Pasig           49
Caloocan        42
Makati          40
Taguig          38
Parañaque       38
Pasay           36
Valenzuela      22
Las Piñas       22
Marikina        22
Mandaluyong     21
Muntinlupa      19
San Juan        11
Malabon         11
Navotas          4
Pateros          2
Name: count, dtype: int64


In [19]:
#Save branch counts

df.to_csv("branches.csv", index=False)
print("Saved branches.csv")

Saved branches.csv


In [21]:
#Calculate distance

import math

# Haversine formula to calculate distance between two coordinates in a sphere.
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth's radius in meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))

In [23]:
#Get pairs

def get_neighbor_pairs(jollibee_df, mcdo_df, distance_m):
    pairs = []
    
    for j, jollibee in jollibee_df.iterrows():
        for m, mcdo in mcdo_df.iterrows():
            dist = haversine(
                jollibee["latitude"], jollibee["longitude"],
                mcdo["latitude"], mcdo["longitude"]
            )
            if dist <= distance_m:
                pairs.append({
                    "jollibee_name": jollibee["name"],
                    "jollibee_address": jollibee["address"],
                    "jollibee_lat": jollibee["latitude"],
                    "jollibee_lng": jollibee["longitude"],
                    "mcdo_name": mcdo["name"],
                    "mcdo_address": mcdo["address"],
                    "mcdo_lat": mcdo["latitude"],
                    "mcdo_lng": mcdo["longitude"],
                    "distance_m": round(dist, 1)
                })
    
    return pd.DataFrame(pairs)

In [24]:
#Get the pairs within 500 meters

pairs_df = get_neighbor_pairs(jollibee_df, mcdo_df, 500)
print(f"Total pairs within 500m: {len(pairs_df)}")
print(pairs_df.head())

Total pairs within 500m: 324
               jollibee_name  \
0                   Jollibee   
1                   Jollibee   
2                   Jollibee   
3                   Jollibee   
4  Jollibee Abad Santos Ave.   

                                    jollibee_address  jollibee_lat  \
0  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
1  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
2  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
3  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
4  3039 Jose Abad Santos, Tondo, Manila, Metro Ma...     14.632186   

   jollibee_lng                       mcdo_name  \
0    121.021055                McDonald's Munoz   
1    121.021055                      McDonald's   
2    121.021055               McDonald's NXTGEN   
3    121.021055                  Jollibee Muñoz   
4    120.980317  McDonald's Abad Santos Hermosa   

                                        mcdo_addre

In [25]:
# One McDonald's branch shows Jollibee. Clean up again.

pairs_df = pairs_df[~pairs_df["mcdo_name"].str.contains("Jollibee", case=False)].reset_index(drop=True)

print(f"Total clean pairs: {len(pairs_df)}")
print(pairs_df.head())

Total clean pairs: 314
               jollibee_name  \
0                   Jollibee   
1                   Jollibee   
2                   Jollibee   
3  Jollibee Abad Santos Ave.   
4                   Jollibee   

                                    jollibee_address  jollibee_lat  \
0  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
1  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
2  1038 Edsa E. Congressional, Bago Bantay, Quezo...     14.657793   
3  3039 Jose Abad Santos, Tondo, Manila, Metro Ma...     14.632186   
4  858 Quezon Ave, Diliman, Quezon City, Metro Ma...     14.631292   

   jollibee_lng                       mcdo_name  \
0    121.021055                McDonald's Munoz   
1    121.021055                      McDonald's   
2    121.021055               McDonald's NXTGEN   
3    120.980317  McDonald's Abad Santos Hermosa   
4    121.017197                      McDonald's   

                                        mcdo_address   m

In [26]:
#Sort pairs by distance

pairs_df = pairs_df.sort_values("distance_m").reset_index(drop=True)
pairs_df.to_csv("pairs.csv", index=False)
print("Saved pairs.csv")
print(pairs_df.head(10))

Saved pairs.csv
                        jollibee_name  \
0  Jollibee - Venice Grand Canal Mall   
1                   Jollibee Commerce   
2       Jollibee Victory Central mall   
3                            Jollibee   
4        Jollibee Unioil Pinaglabanan   
5          Jollibee Metrolane Complex   
6                            Jollibee   
7      Jollibee Pedro Gil st Sta. Ana   
8                   Jollibee PGH Taft   
9                   Jollibee - Anonas   

                                    jollibee_address  jollibee_lat  \
0  Upper McKinley Hl, Taguig, Metro Manila, Phili...     14.533974   
1  Commerce Spectrum Ave., Filinvest, Alabang Mun...     14.420047   
2  Unit # G 04 - 05 Ground Floor, Victory Central...     14.655610   
3  69 Makati Ave, Makati City, 1200 Metro Manila,...     14.562433   
4  CML-HSI Pinaglabanan Place Santolan Road, Halo...     14.604129   
5  102 Metrolane Complex 20th Avenue corner, P. T...     14.621155   
6  315 Epifanio de los Santos Ave, Project

In [29]:
# Clean up the data, and establish a summary with thresholds.

thresholds = [15, 25, 50, 100, 250]

results = []
for distance in thresholds:
    count = len(pairs_df[pairs_df["distance_m"] <= distance])
    jollibee_count = pairs_df[pairs_df["distance_m"] <= distance]["jollibee_name"].nunique()
    results.append({
        "distance_m": distance,
        "unique_jollibee": jollibee_count,
        "total_pairs": count,
        "pct_jollibee": round(jollibee_count / len(jollibee_df) * 100, 1)
    })

summary_df = pd.DataFrame(results)
print(summary_df)

   distance_m  unique_jollibee  total_pairs  pct_jollibee
0          15                1            1           0.3
1          25                6            6           1.8
2          50               22           28           6.6
3         100               49           73          14.8
4         250              103          173          31.0


In [30]:
summary_df.to_csv("distance_summary.csv", index=False)

In [32]:
# Create dataframes for every thresholds for easier mapping.

for distance in [15, 25, 50, 100, 250]:
    filtered = pairs_df[pairs_df["distance_m"] <= distance].reset_index(drop=True)
    print(f"\nWithin {distance}m ({len(filtered)} pairs):")
    print(filtered[["jollibee_name", "jollibee_lat", "jollibee_lng", "mcdo_name", "mcdo_lat", "mcdo_lng", "distance_m"]])


Within 15m (1 pairs):
                        jollibee_name  jollibee_lat  jollibee_lng   mcdo_name  \
0  Jollibee - Venice Grand Canal Mall     14.533974    121.050648  McDonald's   

   mcdo_lat    mcdo_lng  distance_m  
0  14.53407  121.050646        10.7  

Within 25m (6 pairs):
                        jollibee_name  jollibee_lat  jollibee_lng  \
0  Jollibee - Venice Grand Canal Mall     14.533974    121.050648   
1                   Jollibee Commerce     14.420047    121.035961   
2       Jollibee Victory Central mall     14.655610    120.983195   
3                            Jollibee     14.562433    121.028112   
4        Jollibee Unioil Pinaglabanan     14.604129    121.032233   
5          Jollibee Metrolane Complex     14.621155    121.063487   

                 mcdo_name   mcdo_lat    mcdo_lng  distance_m  
0               McDonald's  14.534070  121.050646        10.7  
1  McDonald's Commerce Ave  14.420143  121.035836        17.2  
2  McDonald's Victory Mall  14.655596  

In [34]:
for distance in [15, 25, 50, 100, 250]:
    filtered = pairs_df[pairs_df["distance_m"] <= distance].reset_index(drop=True)
    filtered.to_csv(f"pairs_{distance}m.csv", index=False)

In [36]:
# One location is being mapped outside Metro Manila.
# Check the specific branch outside Metro Manila at the northeast corner
print(df[(df["latitude"] > 14.75) & (df["longitude"] > 121.07)])

        brand      name                                           address  \
100  Jollibee  Jollibee  Quirino Hwy, Caloocan, Metro Manila, Philippines   

      latitude  longitude      city  
100  14.768771  121.08106  Caloocan  


In [37]:
# Drop the branch located outside Metro Manila.
# Branch is listed in Caloocan (within Metro Manila, but coordinates are outside)
df = df.drop(100).reset_index(drop=True)

Total branches: 564


In [42]:
# Change the dataframes to drop the Caloocan branch
pairs_df = pairs_df[pairs_df["jollibee_lat"] != 14.768771].reset_index(drop=True)
pairs_df.to_csv("pairs.csv", index=False)

In [44]:
# Adjust the pairs dataframes for easier mapping
for distance in [15, 25, 50, 100, 250]:
    filtered = pairs_df[pairs_df["distance_m"] <= distance].reset_index(drop=True)
    
    # Extract Jollibee branches
    jollibee_side = filtered[["jollibee_name", "jollibee_address", "jollibee_lat", "jollibee_lng"]].copy()
    jollibee_side.columns = ["name", "address", "latitude", "longitude"]
    jollibee_side["brand"] = "Jollibee"
    
    # Extract McDonald's branches
    mcdo_side = filtered[["mcdo_name", "mcdo_address", "mcdo_lat", "mcdo_lng"]].copy()
    mcdo_side.columns = ["name", "address", "latitude", "longitude"]
    mcdo_side["brand"] = "McDonald's"
    
    # Combine and deduplicate
    combined = pd.concat([jollibee_side, mcdo_side]).drop_duplicates(subset=["latitude", "longitude"]).reset_index(drop=True)
    combined.to_csv(f"map_{distance}m.csv", index=False)
    print(f"Saved map_{distance}m.csv ({len(combined)} branches)")

In [45]:
# Check for any remaining outlier branches
print(df[df["latitude"] > 14.76])
print(df[df["longitude"] < 120.95])

        brand      name                                            address  \
106  Jollibee  Jollibee  Metro Plaza Bldg, Phase 5, Langit Rd, Caloocan...   
119  Jollibee  Jollibee     Langit Rd, Caloocan, Metro Manila, Philippines   

      latitude   longitude      city  
106  14.773205  121.054066  Caloocan  
119  14.776778  121.046520  Caloocan  
          brand                     name  \
295    Jollibee                 Jollibee   
297    Jollibee                 Jollibee   
302    Jollibee                 Jollibee   
428  McDonald's               McDonald's   
559  McDonald's               McDonald’s   
560  McDonald's  McDonald's Hulong Duhat   

                                               address   latitude   longitude  \
295  Gen. Luna St., Cor, Gov. Pascual Ave, Malabon,...  14.669563  120.946880   
297  889-A M. Naval St, Navotas, 1485 Metro Manila,...  14.658562  120.947058   
302  Women's Club, corner Naval St, Malabon, Metro ...  14.675850  120.941753   
428  M. Naval S

In [51]:
#Edit dataframes for mapping. (Previous data include pair branches in one row)
for distance in [15, 25, 50, 100, 250]:
    filtered = pairs_df[pairs_df["distance_m"] <= distance].reset_index(drop=True)
    
    # Extract Jollibee branches
    jollibee_side = filtered[["jollibee_name", "jollibee_address", "jollibee_lat", "jollibee_lng"]].copy()
    jollibee_side.columns = ["name", "address", "latitude", "longitude"]
    jollibee_side["brand"] = "Jollibee"
    
    # Extract McDonald's branches
    mcdo_side = filtered[["mcdo_name", "mcdo_address", "mcdo_lat", "mcdo_lng"]].copy()
    mcdo_side.columns = ["name", "address", "latitude", "longitude"]
    mcdo_side["brand"] = "McDonald's"
    
    # Combine and deduplicate
    combined = pd.concat([jollibee_side, mcdo_side]).drop_duplicates(subset=["latitude", "longitude"]).reset_index(drop=True)
    combined.to_csv(f"map_{distance}m.csv", index=False)

Saved map_15m.csv (2 branches)
Saved map_25m.csv (12 branches)
Saved map_50m.csv (56 branches)
Saved map_100m.csv (133 branches)
Saved map_250m.csv (284 branches)
